# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [ ]:
import pandas as pd
import time

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


In [4]:
df = pd.read_csv('data/bank-additional-full.csv', sep = ';')

In [5]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [31]:
total_0 = len(df[df['duration'] == 0])
total_lte5 = len(df[df['duration'] <= 5])

print(f"total rows with 0 second duration: {total_0}, total rows with duration <= 5s: {total_lte5}")

total rows with 0 second duration: 4, total rows with duration <= 5s: 53


# Initial Analysis
All 21 columns are present, and non have null rows.
There are multiple categorical columns which could be converted to numeric columns via category encoding. For example `job` and `marital` are categorical columns with < 10 values. `month` and `day_of_week` are ordinal data. There are also several boolean features which could be a good fit for one-hot econding.

There are also many columns which are already numeric (int64 or float64).

This should be a good data set for modelling given the number of numeric and numerically encodable features.

Worth noting: one observation shared in the UCI description of the data is that we should discard the `duration` field: 

```
last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
```

However from the original study they included `duration` and found it to be impactful in a different way than was flagged for UCI:

```
Call duration is the most relevant feature, meaning that
longer calls tend increase successes
```

In other words this information could be valuable in a real-world scenario (e.g. as a goal for the sales person performing the pitch to en sure they suprass a certain duration to improve their chances of a successful conversaion).

I also examined the dataset and there are only 4 / 41888 rows which have a duration of exactly 0 and 53 which have a duration <= 5 seconds. For these reasons I will include the `duration` feature.


### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

### Problem 4 Answer: Business Objective
Our business objective is to build a classification model to accurately predict whether a customer will subscribe to a product offering from a bank. For this task we will be using bank data captured from the UCI bank marketing dataset [here](https://archive.ics.uci.edu/dataset/222/bank+marketing). Our target variable name in the original dataset is `y` which is a boolean representing "has the client subscribed a term deposit?".

Our apprach will be to evaluate multiple classification models to determine which one is the best at predicting subscription outcomes from the marketing campaign. The specififc models we will eavaluate are Logistic Regression, KNN, Decision Tres, and Support Vector Machines (SVM).

---

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [42]:
df_encoded = df.copy()

# --- Target: encode y as binary ---
df_encoded['y'] = df_encoded['y'].map({'yes': 1, 'no': 0})

# --- Ordinal features: preserve ordering ---
education_order = [['unknown', 'illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
                    'high.school', 'professional.course', 'university.degree']]
month_order = [['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']]
day_order = [['mon', 'tue', 'wed', 'thu', 'fri']]

df_encoded['education'] = OrdinalEncoder(categories=education_order).fit_transform(df_encoded[['education']])
df_encoded['month'] = OrdinalEncoder(categories=month_order).fit_transform(df_encoded[['month']])
df_encoded['day_of_week'] = OrdinalEncoder(categories=day_order).fit_transform(df_encoded[['day_of_week']])

# --- Binary categorical features: yes/no/unknown → 1/0/-1 ---
for col in ['default', 'housing', 'loan']:
    df_encoded[col] = df_encoded[col].map({'yes': 1, 'no': 0, 'unknown': -1})

# contact has no unknown category
df_encoded['contact'] = df_encoded['contact'].map({'cellular': 1, 'telephone': 0})

# --- Nominal categorical features: one-hot encode ---
df_encoded = pd.get_dummies(df_encoded, columns=['job', 'marital', 'poutcome'], drop_first=False)

X = df_encoded.drop(columns=['y'])
y = df_encoded['y']

print(f"Feature matrix shape: {X.shape}")
print(f"\nTarget distribution:\n{y.value_counts()}")
X.head()

Feature matrix shape: (41188, 36)

Target distribution:
y
0    36548
1     4640
Name: count, dtype: int64


,age,education,default,housing,loan,contact,month,day_of_week,duration,campaign,...,job_technician,job_unemployed,job_unknown,marital_divorced,marital_married,marital_single,marital_unknown,poutcome_failure,poutcome_nonexistent,poutcome_success
0,56,2.0,0,0,0,0,4.0,0.0,261,1,...,False,False,False,False,True,False,False,False,True,False
1,57,5.0,-1,0,0,0,4.0,0.0,149,1,...,False,False,False,False,True,False,False,False,True,False
2,37,5.0,0,1,0,0,4.0,0.0,226,1,...,False,False,False,False,True,False,False,False,True,False
3,40,3.0,0,0,0,0,4.0,0.0,151,1,...,False,False,False,False,True,False,False,False,True,False
4,56,5.0,0,0,1,0,4.0,0.0,307,1,...,False,False,False,False,True,False,False,False,True,False


### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [ ]:
# -- Classes are imbalanced -- stratify to ensure both splits maintain same ratio.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain target distribution:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTest target distribution:\n{y_test.value_counts(normalize=True).round(3)}")



Train target distribution:
y
0    0.887
1    0.113
Name: proportion, dtype: float64

Test target distribution:
y
0    0.887
1    0.113
Name: proportion, dtype: float64


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [45]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

baseline_train_acc = dummy.score(X_train, y_train)
baseline_test_acc  = dummy.score(X_test, y_test)

print(f"Baseline (most frequent) train accuracy: {baseline_train_acc:.4f}")
print(f"Baseline (most frequent) test accuracy:  {baseline_test_acc:.4f}")

Baseline (most frequent) train accuracy: 0.8873
Baseline (most frequent) test accuracy:  0.8874


### Problem 7 Answer:
Our classifier must score above 0.8874 on test accuracy in order to outperform the baseline.


### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipe.fit(X_train, y_train)

### Problem 9: Score the Model

What is the accuracy of your model?

In [14]:
lr_train_acc = lr_pipe.score(X_train, y_train)
lr_test_acc  = lr_pipe.score(X_test, y_test)

print(f"Logistic Regression train accuracy: {lr_train_acc:.4f}")
print(f"Logistic Regression test accuracy:  {lr_test_acc:.4f}")

Logistic Regression train accuracy: 0.9101
Logistic Regression test accuracy:  0.9122


### Problem 9 Answer:

The initial LR model has an accuracy of 91.22% on the test data (above the 89.94% baseline on test data).

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [19]:
lr_pipe = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(random_state=42))])
knn_pipe = Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier())])
dt_pipe = Pipeline([('scaler', StandardScaler()), ('model', DecisionTreeClassifier(random_state=42))])
svm_pipe = Pipeline([('scaler', StandardScaler()), ('model', SVC(random_state=42))])

models = [
    ('Logistic Regression', lr_pipe),
    ('KNN', knn_pipe),
    ('Decision Tree', dt_pipe),
    ('SVM', svm_pipe),
]

results = []
for name, pipe in models:
    start = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - start

    results.append({
        'Model': name,
        'Train Time (s)': round(train_time, 4),
        'Train Accuracy': round(pipe.score(X_train, y_train), 4),
        'Test Accuracy': round(pipe.score(X_test, y_test), 4),
    })

results_df = pd.DataFrame(results).set_index('Model')
results_df

,Train Time (s),Train Accuracy,Test Accuracy
Model,,,
Logistic Regression,0.1284,0.9101,0.9122
KNN,0.0381,0.9238,0.9016
Decision Tree,0.1446,1.0000,0.8906
SVM,5.4462,0.9179,0.9122


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

##### Questions